# 05 – Feature Engineering

**Project:** Diabetes Prediction ML  
**Seminar:** Advanced Applied Data Science – Goethe University Frankfurt  
**Dataset:** CDC BRFSS 2015 – Diabetes Health Indicators (UCI #891)  
**Target:** `Diabetes_binary` (0 = No Diabetes, 1 = Prediabetes/Diabetes)  
**CRISP-DM phase:** 3 – Data Preparation · Feature Engineering

## Purpose

This notebook builds the candidate engineered features for the diabetes classifier and then screens them: it decides, by cross-validated PR-AUC against the raw-21 baseline band from NB04, whether any engineered block earns a place in the modelling pool. Section 2 constructs the candidates; Sections 3–5 run the controlled screening and assemble the output pool.

The design is experiment-first: every feature is added **additively, without dropping anything**, so feature *selection* — decided in Section 3 by cross-validated PR-AUC against the raw-21 band, never by univariate filters — can choose arbitrary subsets. Lasso-path stability, permutation importance, and a selection-optimism check (Sections 3.7–3.8) are reported only to triangulate and explain that decision — they never gate it. The expectation, set by the small linear↔tree gap measured in NB04 (PR-AUC 0.404 vs. 0.434), is deliberately modest: **near-zero gain for gradient-boosted trees** (they reconstruct these functions of the raw columns internally) and a **modest gain for Logistic Regression** from explicit non-linearity.

All engineered features are **row-wise deterministic** — computed per row with no target and no fitted statistic — so they carry no leakage and may be built before the split. The only learnable step, in-fold feature scaling, lives in the evaluation pipeline (as in NB04), not here.

## 1. Setup

In [1]:
import json, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd

# Walk up from the current dir to the repo root (the folder containing notebooks/),
# so the imports below work no matter where the notebook is launched from.
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir())
DATA_DIR    = PROJECT_ROOT / "data" / "processed"
RESULTS_CSV = PROJECT_ROOT / "outputs" / "results.csv"

# Put the repo root on sys.path so `src` is importable regardless of cwd.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.utils import log_result, make_cv
from src.features import build_candidates, WHO_LABELS, WHO_BINS  # WHO_LABELS used to derive the one-hot list

# Canonical feature list + seed from NB03 (single source of truth).
with open(DATA_DIR / "feature_meta.json") as f:
    feature_meta = json.load(f)
ALL_FEATURES = feature_meta["all_features"]
SEED = feature_meta["seed"]

# Raw 21-feature split; reindex to canonical column order (identical to NB04) so the
# group hash — and therefore the CV fold partition — is bit-identical to NB04.
X_train = pd.read_parquet(DATA_DIR / "X_train.parquet")[ALL_FEATURES]
y_train = pd.read_parquet(DATA_DIR / "y_train.parquet").squeeze()

print(f"X_train: {X_train.shape}  |  features: {len(ALL_FEATURES)}  |  seed: {SEED}")
print(f"positive prevalence: {y_train.mean():.4f}")

X_train: (202943, 21)  |  features: 21  |  seed: 42
positive prevalence: 0.1393


## 2. Feature engineering — definitions & construction

Each block below adds its features **without removing the originals** (composable, no dropping), so the selection step can test any subset. Rationale is kept to one line per feature with its source — this is a feature catalogue, not a literature review.

Two structural notes carried over from the planning:
- **`BMI_squared` is additive** (kept alongside raw `BMI`) so a linear model can fit the J-shaped BMI–diabetes risk; `BMI_capped`, `BMI_cat` and the one-hot WHO classes are **swap alternatives**, compared against each other (and against raw BMI + `BMI_squared`) in the encoding ablation later — not meant to be used together.
- The **linear-sum composites** (e.g. `comorbidity_count`, `ses_index`) are **linearly redundant for Logistic Regression** — they are exact sums of raw columns it already has. **Interactions are not**: a product is structure a linear model cannot form from its inputs, so it is genuinely new for LogReg (the screen later shows a real, if small, interaction gain). Both are **largely free for trees to reconstruct**, and are built anyway so the screen can confirm this empirically and for their interpretive value.

**`BMI_capped` bounds [18, 50] are fixed clinical bounds, not data-derived percentiles** — hence leakage-free and not an in-fold step (reconciling the NB02 leakage discussion; note `BMI_capped` lost the ablation).

### 2.1 BMI re-encodings

These re-encodings target Logistic Regression. A tree finds BMI thresholds internally, so a re-encoding cannot give it structure it lacks: `BMI_squared` is a monotone transform trees are invariant to, while capping and binning are coarsenings that — for a tree — can only discard resolution, never add it. For the linear model, by contrast, an explicit non-linear BMI term is genuinely new structure. Exact definitions (from `src/features.py`):

- `BMI_squared` = `BMI ** 2` — additive non-linearity capturing the over-proportional (J-shaped) risk rise at high BMI (Tirosh et al. 2011, *NEJM*); kept **alongside** raw `BMI`.
- `BMI_capped` = `BMI.clip(18, 50)` — swap alternative; fixed clinical bounds (not data-derived percentiles), so leakage-free.
- `BMI_cat` = WHO class `0–6` of `BMI` (cut-points shown below) — swap alternative and the building block for the composite scores.
- `BMI_class_*` = one-hot of `BMI_cat`, dropping class 1 (normal) as the reference category — swap alternative.

WHO classes use the standard cut-points (WHO TRS 894, 2000). Scaling/centering is handled in-fold by the evaluation pipeline, so the columns here stay row-wise deterministic. Construction is factored into `src/features.py` so NB08 applies the identical transform to the frozen test set; the cell below prints the exact cut-points used.

In [2]:
# The WHO cut-points that define BMI_cat and the BMI_class_* one-hot.
# Derived from src/features.py (single source of truth), left-closed [lo, hi)
# to match the right=False convention we standardise on in _bmi_cat.
who_names = ["underweight", "normal", "overweight", "obese-I", "obese-II", "obese-III", "super-obese"]
for lab, lo, hi, name in zip(WHO_LABELS, WHO_BINS[:-1], WHO_BINS[1:], who_names):
    print(f"class {lab} ({name:11s}): {lo:>5} <= BMI < {hi}")

class 0 (underweight):     0 <= BMI < 18.5
class 1 (normal     ):  18.5 <= BMI < 25
class 2 (overweight ):    25 <= BMI < 30
class 3 (obese-I    ):    30 <= BMI < 35
class 4 (obese-II   ):    35 <= BMI < 40
class 5 (obese-III  ):    40 <= BMI < 50
class 6 (super-obese):    50 <= BMI < 999


### 2.2 Zero-inflation transforms (MentHlth, PhysHlth)

`MentHlth` and `PhysHlth` (bad-health days in the last 30) are strongly **zero-inflated** (observed in NB02): a large spike at 0 plus a long right tail. A single linear term cannot express both the "any bad days at all" discontinuity and the "how many" gradient, so each count is split into two pieces, following the **hurdle model** (Mullahy 1986, *J Econometrics*). Definitions (from `src/features.py`):

- `MentHlth_any` = `(MentHlth > 0).astype(int)` — hurdle indicator (extensive margin: any vs. none). **Additive**: the zero/non-zero discontinuity is genuinely new structure the raw count cannot express linearly.
- `MentHlth_log` = `log1p(MentHlth)` — compresses the count's right-skew for the linear model (intensive margin). A **re-encoding** of the same count's scale (swap-style, analogous to the BMI encodings).
- `PhysHlth_any`, `PhysHlth_log` — identical transforms on `PhysHlth`.

Raw `MentHlth`/`PhysHlth` stay in the raw-21; the block is screened **additively** (`raw + any + log`), which is the generous ceiling for what these transforms can add — so no separate swap ablation is needed. Trees are invariant to the monotone `log1p`, so no tree gain is expected.

### 2.3 Clinical composites

Row-wise sums **adapted** from validated, biomarker-free risk scores, restricted to the BRFSS-available subset. These are adaptations, **not the validated scores** — equal-weighted sums of a variable subset, so they capture direction, not the calibrated point-weights of the originals. Definitions (from `src/features.py`):

- `comorbidity_count` = `HighBP + HighChol + Stroke + HeartDiseaseorAttack` — unweighted cardiometabolic burden (Muhammad et al. 2025, top SHAP predictor on Tennessee BRFSS). *Linearly redundant for LogReg.*
- `findrisc_lite` = `BMI_cat + Age + (1−PhysActivity) + (1−Fruits) + (1−Veggies)` — FINDRISC-adapted (Lindström & Tuomilehto 2003, *Diabetes Care*); subset without waist circumference / glucose history, unweighted.
- `ada_risk_proxy` = `Age + BMI_cat + HighBP + (1−PhysActivity) + Sex` — ADA Diabetes Risk Test subset (Bang et al. 2009, *Ann Intern Med*); without family history, unweighted.
- `healthy_lifestyle` = `PhysActivity + Fruits + Veggies + (1−Smoker) + (1−HvyAlcoholConsump)` — AHA *Life's Simple 7* behaviours (Lloyd-Jones et al. 2010, *Circulation*); behavioural subset only. *Linearly redundant for LogReg.*
- `ses_index` = `Education + Income` — socioeconomic position (Agardh et al. 2011, *Int J Epidemiol*). *Linearly redundant for LogReg.*
- `healthcare_access_index` = `CholCheck + AnyHealthcare + (1−NoDocbcCost)` — system-contact / **diagnosis-probability** proxy, ties to the label-noise thesis (Andersen & Newman 1973). *Linearly redundant for LogReg.*

### 2.4 Interactions

Explicit products give the linear model interaction structure it **cannot form from its inputs** — a product is not a linear combination of its factors, so for LogReg this is genuinely new (not redundant). Trees build the same structure natively via splits, so the expected tree gain is ≈ 0. Built on raw `BMI` and `Age` (note `Age` is the 13-level 5-year band, so `BMI_x_Age` is BMI × age-band, not BMI × years); the whole matrix is standardised in-fold. Definitions (from `src/features.py`):

- `BMI_x_HighBP` = `BMI * HighBP` — obesity × hypertension synergy (Landsberg et al. 2013).
- `BMI_x_Age` = `BMI * Age` — age-modified adiposity effect (Janssen et al. 2005).
- `Age_x_HighBP` = `Age * HighBP` — age-dependent hypertension effect (rationale-based, no single source).
- `HighBP_x_HighChol` = `HighBP * HighChol` — metabolic-syndrome co-occurrence; product of two binaries = logical AND (IDF 2006).

### 2.5 Self-rated-health flag

`poor_health` = `(GenHlth >= 4).astype(int)` — marks fair/poor self-rated health on the 5-level `GenHlth` scale (1 excellent … 5 poor). Self-rated health is a validated predictor of morbidity and mortality (Idler & Benyamini 1997, *J Health Soc Behav*). It gives the linear model a **threshold non-linearity** — a jump at fair/poor that a single linear `GenHlth` term cannot represent (so it is *not* linearly redundant). Redundant for trees, which can split `GenHlth` at that cut-point directly.

### 2.6 Build the candidate matrix & sanity checks

`build_candidates` applies every block additively and returns the raw 21 features plus all engineered candidates — nothing dropped. The selection step (next) chooses subsets from this matrix; here we assert it builds cleanly: no NaNs, all 21 raw columns kept, and the expected 24 engineered columns — a guard that catches any drift in `src/features.py` (e.g. a missing one-hot class).

In [3]:
X_cand = build_candidates(X_train)
engineered = [c for c in X_cand.columns if c not in ALL_FEATURES]

# sanity guards: nothing dropped, a fixed engineered set is added, no NaNs
assert set(ALL_FEATURES).issubset(X_cand.columns), "a raw feature was dropped"
assert len(engineered) == 24, f"expected 24 engineered columns, got {len(engineered)}"
assert X_cand.isna().sum().sum() == 0, "unexpected NaNs in candidate matrix"

print(f"raw features:        {len(ALL_FEATURES)}")
print(f"engineered features: {len(engineered)}")
print(f"candidate matrix:    {X_cand.shape}  (no NaNs, all raw kept)")
print("engineered columns:")
for c in engineered:
    print("  -", c)

raw features:        21
engineered features: 24
candidate matrix:    (202943, 45)  (no NaNs, all raw kept)
engineered columns:
  - BMI_squared
  - BMI_capped
  - BMI_cat
  - BMI_class_0
  - BMI_class_2
  - BMI_class_3
  - BMI_class_4
  - BMI_class_5
  - BMI_class_6
  - MentHlth_any
  - MentHlth_log
  - PhysHlth_any
  - PhysHlth_log
  - comorbidity_count
  - findrisc_lite
  - ada_risk_proxy
  - healthy_lifestyle
  - ses_index
  - healthcare_access_index
  - BMI_x_HighBP
  - BMI_x_Age
  - Age_x_HighBP
  - HighBP_x_HighChol
  - poor_health


## 3. Controlled screening — does any engineered block beat the raw-21 band?

Section 2 only *built* the candidates. This section decides which of them earn a place in the
candidate pool, using a pre-specified design:

- **Two instruments that bracket the representation axis.** LightGBM reconstructs monotone
  transforms and interactions internally; Logistic Regression can only form linear combinations of
  the columns it is given. A feature that helps LogReg adds linearly-unavailable structure; one that
  does not help LightGBM is already implicit in the raw columns. Together they span "needs it
  explicitly" ↔ "builds it itself", so a single screening pass produces candidates for the wider
  model space (whether a *distance-based* family has its own gains is an NB06 question, not a second
  screen).
- **Same criterion on both:** cross-validated PR-AUC against *each model's own* raw-21 result, on the
  identical group-aware folds, judged by **paired per-fold differences**.
- **Coarse-to-fine:** Stage 0 bracket → Stage 1 block ablation (first-pass selection) → Stage 1b BMI encoding ablation → direct candidates with a **conditional** contribution test (where the final pool is fixed) → triangulation that is strictly descriptive and never gates.

Expectation set by the 0.030 linear↔tree gap from NB04: **≈ 0 for LightGBM, a small gain at most for
LogReg.** If nothing beats the band, raw-21 stays — that is a result, not a failure.

### 3.1 Shared mechanics (identical to NB04)

Groups are the **raw-21 profile hash** (computed in NB03/NB04) and are *never* recomputed on the
engineered columns — profile identity (the ~14 % exact duplicates) is a property of the raw features,
and re-hashing on 45 columns would shift the folds and break the duplicate-leakage guard. The raw-21
reference band is recomputed here so its folds are bit-identical to every set tested below (same
`cv`, `SEED`, `groups`), which is exactly what makes the paired delta valid.

In [4]:
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier

# groups = raw-21 profile hash, reindexed to canonical order (identical to NB04)
groups = pd.util.hash_pandas_object(X_train, index=False).values
cv = make_cv(seed=SEED)
SCORING = ["average_precision", "roc_auc"]
# scale_pos_weight: global (not fold-specific), identical to NB04
spw = float((y_train == 0).sum() / (y_train == 1).sum())

def make_logreg():
    return ImbPipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED)),
    ])

def make_lgbm():
    return LGBMClassifier(scale_pos_weight=spw, verbose=-1, random_state=SEED, n_jobs=-1)

INSTRUMENTS = {"LogReg": make_logreg, "LightGBM": make_lgbm}

X_cand = build_candidates(X_train)
assert X_cand.isna().sum().sum() == 0
RAW21 = list(ALL_FEATURES)

print(f"candidate matrix: {X_cand.shape}  |  unique groups: {len(set(groups)):,}  |  spw: {spw:.3f}")

candidate matrix: (202943, 45)  |  unique groups: 182,337  |  spw: 6.177


In [5]:
def evaluate(name, columns, instrument, log=True):
    # one group-aware CV pass; folds are identical across calls (same cv/SEED/groups),
    # which is what makes the paired delta in 3.2 valid
    est = INSTRUMENTS[instrument]()
    sc = cross_validate(est, X_cand[columns], y_train, groups=groups,
                        cv=cv, scoring=SCORING, n_jobs=-1)
    pr, roc = sc["test_average_precision"], sc["test_roc_auc"]
    if log:
        log_result(RESULTS_CSV, notebook="nb05", model=instrument,
                   pr_auc_mean=float(pr.mean()), pr_auc_std=float(pr.std()),
                   roc_auc_mean=float(roc.mean()), feature_set=name,
                   imbalance="class_weight" if instrument == "LogReg" else "scale_pos_weight",
                   n_folds=5, seed=SEED, pr_auc_folds=pr, roc_auc_folds=roc)
    return {"name": name, "instrument": instrument, "pr": pr, "roc": roc,
            "pr_mean": float(pr.mean()), "pr_std": float(pr.std())}

# raw-21 reference per instrument; the std should now match NB04 (±0.0076 / ±0.0091),
# which confirms the reindex made the fold partition identical to NB04.
ref = {}
for inst in INSTRUMENTS:
    r = evaluate("raw21", RAW21, inst)
    ref[inst] = r
    nb04 = "0.404 \u00b1 0.0076" if inst == "LogReg" else "0.434 \u00b1 0.0091"
    print(f"{inst:9s} raw21  PR-AUC = {r['pr_mean']:.4f} +/- {r['pr_std']:.4f}   (NB04 band: {nb04})")

LogReg    raw21  PR-AUC = 0.4041 +/- 0.0076   (NB04 band: 0.404 ± 0.0076)
LightGBM  raw21  PR-AUC = 0.4343 +/- 0.0091   (NB04 band: 0.434 ± 0.0091)


### 3.2 The pre-specified decision rule

For variant *v* and model *m* we have five paired per-fold PR-AUCs from the **same** folds:
`a₁…a₅` (variant) and `b₁…b₅` (raw-21, same model). The per-fold delta is `Δₖ = aₖ − bₖ`. Pairing
shrinks the variance — `Var(Δ) = Var(a) + Var(b) − 2·Cov(a,b)`, and shared folds make `Cov(a,b)`
large — so a `+0.003` delta can be real measured against the spread of the Δₖ, not against the
±0.009 of the individual bands.

**The rule is fixed before any delta is inspected** (two separate questions, both must hold to
retain an *additive* feature):

- **Real?** positive Δ in **≥ 4 of 5 folds** *and* `mean(Δ) ≥ 2·SE(Δ)`, with `SE(Δ)=sd(Δ)/√5`.
- **Worth it?** `mean(Δ) ≥ C_PRACTICAL = 0.005` — a pre-set practical-relevance floor (≈ ½ the
  per-fold std, ≈ 1/6 of the 0.030 linear↔tree gap). Below it the gain does not justify the
  collinearity/interpretability cost for LogReg; for trees this is essentially always "no", and that
  is the expected, correct result.

**On `2·SE` and `C_PRACTICAL` (stated, not hidden).** `SE(Δ) = sd(Δ)/√5` is a rough noise-margin heuristic, **not a significance test**. It assumes the five folds are independent, but they are not: the training sets overlap (any two share 3/5 of the data), so the Δₖ are positively correlated across folds and `sd/√5` *understates* the true SE — there is no unbiased estimator of k-fold variance (Bengio & Grandvalet 2004). (This across-fold dependence is a *different* covariance from the within-fold `Cov(a,b)` that pairing exploits above: fold-sharing tightens each `Δₖ` but inflates our confidence about their mean.) The `2·SE` bar is therefore deliberately lenient; the real backstops are the **≥ 4/5 sign requirement** and the single unbiased test-set evaluation in NB08. We report all five `Δₖ`, the sign count, mean, sd and SE — never a p-value (a Wilcoxon test at n = 5 cannot reach two-sided p < 0.05; see NB04). `C_PRACTICAL = 0.005` is likewise a **judgment** floor (≈ ½ the per-fold std, ≈ 1/6 of the 0.030 linear↔tree gap), not an objective constant.

In [6]:
C_PRACTICAL = 0.005   # pre-specified practical-relevance floor (NOT tuned post hoc)
SIGN_MIN    = 4       # pre-specified: positive delta required in >= 4 of 5 folds

def paired_delta(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    mean = float(d.mean()); sd = float(d.std(ddof=1)); se = sd / np.sqrt(len(d))
    n_pos = int((d > 0).sum())
    real  = (n_pos >= SIGN_MIN) and (mean >= 2 * se)   # NB: se optimistic (Bengio & Grandvalet 2004)
    worth = mean >= C_PRACTICAL
    return {"deltas": d, "mean": mean, "sd": sd, "se": se, "n_pos": n_pos,
            "real": bool(real), "worth": bool(worth), "retain": bool(real and worth)}

def screen(name, columns, instrument):
    r = evaluate(name, columns, instrument); d = paired_delta(r["pr"], ref[instrument]["pr"])
    r.update(d)
    flag = "RETAIN" if d["retain"] else ("real, not worth" if d["real"] else "no robust gain")
    print(f"{instrument:9s} {name:36s} PR {r['pr_mean']:.4f}  d={d['mean']:+.4f}  "
          f"sign {d['n_pos']}/5  2SE={2*d['se']:.4f}  -> {flag}")
    return r

### 3.3 Stage 0 — the bracket (`raw21` ↔ `raw21 + all_additive`)

Ceiling / stress test only: "does *anything* help?". `all_additive` contains **only additive terms**
(the four blocks + `BMI_squared`, raw BMI kept) — the swap encodings live in 1b, never here. A null
result says nothing about *which* feature; a positive one is diluted. Useless as selection, useful as
an upper bound.

In [7]:
BLOCKS = {
    "count_hurdle": ["MentHlth_any", "MentHlth_log", "PhysHlth_any", "PhysHlth_log"],
    "composites":   ["comorbidity_count", "findrisc_lite", "ada_risk_proxy",
                     "healthy_lifestyle", "ses_index", "healthcare_access_index"],
    "interactions": ["BMI_x_HighBP", "BMI_x_Age", "Age_x_HighBP", "HighBP_x_HighChol"],
    "flag":         ["poor_health"],
}
ADDITIVE_ENG = [c for cols in BLOCKS.values() for c in cols] + ["BMI_squared"]
ALL_ADDITIVE = RAW21 + ADDITIVE_ENG

bracket = {inst: screen("raw21 + all_additive", ALL_ADDITIVE, inst) for inst in INSTRUMENTS}

LogReg    raw21 + all_additive                 PR 0.4203  d=+0.0161  sign 5/5  2SE=0.0016  -> RETAIN
LightGBM  raw21 + all_additive                 PR 0.4338  d=-0.0005  sign 3/5  2SE=0.0012  -> no robust gain


#### Interpretation — Stage 0

Adding all 16 additive terms (the four blocks + `BMI_squared`) lifts LogReg by Δ = +0.0161 (5/5 folds, 2SE = 0.0016) — a robust, practically relevant effect (PR-AUC 0.4041 → 0.4203). LightGBM shows Δ = −0.0005 (3/5 folds nominally positive, but the mean is slightly negative and far inside the noise band, 2SE = 0.0012) — no robust gain.

Why the contrast? LogReg can only form linear combinations of the columns it is given, so the explicitly constructed non-linearities (BMI², hurdle indicators, composite scores) are genuinely new information. LightGBM reconstructs monotone transforms, thresholds, and interactions internally from the raw columns — the engineered columns add no structure it cannot already build. This is exactly the headroom implied by the 0.030 linear↔tree gap from NB04 (LogReg 0.404, LightGBM 0.434); the bracket confirms it empirically. Note this is a ceiling/stress test only — it shows *something* helps LogReg, not *which* block; attribution happens in Stage 1.

### 3.4 Stage 1 — block ablation (the actual selection)

Each `+block` against raw-21, per instrument. Blocks are the sweet spot between single-feature
(drowns small effects in the ±0.009 noise, is quasi-univariate) and all-at-once (can't attribute).
Retain/drop is decided **here**, by the rule in 3.2.

In [8]:
block_results = {inst: {} for inst in INSTRUMENTS}
for inst in INSTRUMENTS:
    print(f"--- {inst} ---")
    for bname, cols in BLOCKS.items():
        block_results[inst][bname] = screen(f"raw21 + {bname}", RAW21 + cols, inst)
    print()

--- LogReg ---
LogReg    raw21 + count_hurdle                 PR 0.4045  d=+0.0003  sign 5/5  2SE=0.0002  -> real, not worth
LogReg    raw21 + composites                   PR 0.4145  d=+0.0104  sign 5/5  2SE=0.0011  -> RETAIN
LogReg    raw21 + interactions                 PR 0.4085  d=+0.0044  sign 5/5  2SE=0.0006  -> real, not worth
LogReg    raw21 + flag                         PR 0.4038  d=-0.0004  sign 0/5  2SE=0.0001  -> no robust gain

--- LightGBM ---
LightGBM  raw21 + count_hurdle                 PR 0.4343  d=-0.0000  sign 0/5  2SE=0.0000  -> no robust gain
LightGBM  raw21 + composites                   PR 0.4339  d=-0.0004  sign 2/5  2SE=0.0008  -> no robust gain
LightGBM  raw21 + interactions                 PR 0.4337  d=-0.0005  sign 1/5  2SE=0.0018  -> no robust gain
LightGBM  raw21 + flag                         PR 0.4343  d=+0.0000  sign 0/5  2SE=0.0000  -> no robust gain



#### Interpretation — Block ablation

**LogReg:** Only the `composites` block clears both hurdles: Δ = +0.0104, 5/5 folds, above the practical floor (> 0.005) → **RETAIN**. Both `interactions` (Δ = +0.0044, 5/5) and `count_hurdle` (Δ = +0.0003, 5/5, but tiny) are **real but not worth it** — directionally consistent across all five folds, yet below the 0.005 floor. Only `flag` shows **no robust gain** (Δ = −0.0004, 0/5): the threshold dummy adds nothing the linear `GenHlth` term does not already capture.

**LightGBM:** No block achieves a robust gain — all deltas are near zero or negative, sign-count ≤ 2/5. This confirms the expectation: hurdle transforms, composite scores, and interactions are all redundant for gradient boosting.

**Sub-additivity warning:** Block deltas do not add up — `composites` contains `BMI_cat`, which captures the same BMI non-linearity as `BMI_squared`, so the gains overlap. This is why the next section measures the *conditional* contribution of `composites` *given* the winning BMI encoding, rather than treating it as an independent addend.

### Sensitivity: retention at different C_PRACTICAL thresholds

We re-examine block retention at C ∈ {0.003, 0.005, 0.010} to show how sensitive the keep/drop decision is to the choice of practical floor.

In [9]:
print("=== Sensitivity: retention decision at different C_PRACTICAL thresholds ===")
for C_test in [0.003, 0.005, 0.010]:
    retained_lr = [b for b, r in block_results["LogReg"].items()
                   if r["real"] and r["mean"] >= C_test]
    retained_lgbm = [b for b, r in block_results["LightGBM"].items()
                     if r["real"] and r["mean"] >= C_test]
    print(f"  C = {C_test:.3f}: LogReg retains {str(retained_lr or 'none'):<30}  LightGBM retains {retained_lgbm or 'none'}")

=== Sensitivity: retention decision at different C_PRACTICAL thresholds ===
  C = 0.003: LogReg retains ['composites', 'interactions']  LightGBM retains none
  C = 0.005: LogReg retains ['composites']                  LightGBM retains none
  C = 0.010: LogReg retains ['composites']                  LightGBM retains none


#### Interpretation — Sensitivity

The keep/drop decision is **not knife-edge** at the chosen 0.005: `composites` is retained across 0.003–0.010. `interactions` is the genuine borderline case — in at 0.003, out at 0.005 — which is exactly why it is reported as "real, not worth" rather than retained. LightGBM retains nothing at any threshold.

More importantly, the *final* pool is even more robust than this Stage-1 view suggests: it rests on `BMI_squared` (+0.0148, which clears any reasonable floor), and `composites` is re-tested **conditionally** on `BMI_squared` in the next section — where its contribution collapses to ≈ 0. So the final headline pool (raw21 + BMI²) does not hinge on the exact value of `C_PRACTICAL`.

### 3.5 Stage 1b — BMI encoding ablation

`BMI_squared` is additive (tested in Stage 1 via `all_additive`); `BMI_capped`, `BMI_cat` and the WHO
one-hot are **mutually exclusive** swap encodings — throwing them in together would give LogReg
perfectly collinear columns. Head-to-head, one per run.

For LightGBM, large gains from BMI re-encodings are not expected, because tree models can learn thresholds internally; therefore BMI encoding is screened mainly for LogReg, and run on LightGBM only as a confirmation.

The winning encoding feeds the final pool.

In [10]:
NO_BMI = [c for c in RAW21 if c != "BMI"]
WHO_OH = ["BMI_class_0", "BMI_class_2", "BMI_class_3", "BMI_class_4", "BMI_class_5", "BMI_class_6"]
bmi_sets = {
    "raw21 (BMI raw, ref)":              RAW21,
    "raw21 + bmi_squared":               RAW21 + ["BMI_squared"],
    "raw21 [BMI->capped]":               NO_BMI + ["BMI_capped"],
    "raw21 [BMI->cat]":                  NO_BMI + ["BMI_cat"],
    "raw21 [BMI->who_onehot]+bmi_sq":    NO_BMI + WHO_OH + ["BMI_squared"],
}
print("--- LogReg BMI ablation ---")
bmi_results = {s: screen(s, cols, "LogReg") for s, cols in bmi_sets.items()}

--- LogReg BMI ablation ---
LogReg    raw21 (BMI raw, ref)                 PR 0.4041  d=+0.0000  sign 0/5  2SE=0.0000  -> no robust gain
LogReg    raw21 + bmi_squared                  PR 0.4189  d=+0.0148  sign 5/5  2SE=0.0018  -> RETAIN
LogReg    raw21 [BMI->capped]                  PR 0.4166  d=+0.0125  sign 5/5  2SE=0.0011  -> RETAIN
LogReg    raw21 [BMI->cat]                     PR 0.4151  d=+0.0110  sign 5/5  2SE=0.0014  -> RETAIN
LogReg    raw21 [BMI->who_onehot]+bmi_sq       PR 0.4146  d=+0.0105  sign 5/5  2SE=0.0021  -> RETAIN


In [11]:
print("\n--- LightGBM BMI ablation (confirmation; large gains not expected for trees) ---")
bmi_results_lgbm = {s: screen(s, cols, "LightGBM") for s, cols in bmi_sets.items()}


--- LightGBM BMI ablation (confirmation; large gains not expected for trees) ---
LightGBM  raw21 (BMI raw, ref)                 PR 0.4343  d=+0.0000  sign 0/5  2SE=0.0000  -> no robust gain
LightGBM  raw21 + bmi_squared                  PR 0.4343  d=+0.0000  sign 0/5  2SE=0.0000  -> no robust gain
LightGBM  raw21 [BMI->capped]                  PR 0.4334  d=-0.0008  sign 1/5  2SE=0.0007  -> no robust gain
LightGBM  raw21 [BMI->cat]                     PR 0.4312  d=-0.0030  sign 0/5  2SE=0.0009  -> no robust gain
LightGBM  raw21 [BMI->who_onehot]+bmi_sq       PR 0.4338  d=-0.0005  sign 1/5  2SE=0.0006  -> no robust gain


#### Interpretation — BMI ablation

**LogReg:** All four alternatives beat raw BMI (Δ > 0, 5/5 folds): `bmi_squared` +0.0148, `bmi_capped` +0.0125, `bmi_cat` +0.0110, `who_onehot+bmi_sq` +0.0105. The clear winner is `raw21 + bmi_squared` (Δ = +0.0148, 2SE = 0.0018) — the simplest continuous encoding, modelling the J-shaped BMI–diabetes risk curve (Tirosh et al. 2011) directly while keeping raw BMI, without the information loss of binning or clipping. `bmi_capped` scores slightly lower (+0.0125) because clipping at 50 softens rare but clinically meaningful extremes. The categorical encodings (`bmi_cat` +0.0110, `WHO one-hot` +0.0105) are also real.

**LightGBM:** No encoding helps the tree (no robust gain, sign-count ≤ 1/5) — but they are not all identical. The monotone `bmi_squared` is exactly invariant (Δ = +0.0000), while the coarsenings slightly *hurt*, most clearly `bmi_cat` (Δ = −0.0030, well outside the ±2SE noise). This is the empirical confirmation of the corrected framing: re-encodings are **not** invariant "by construction" — `bmi_capped`, `bmi_cat` and the WHO one-hot are clamps/coarsenings that *can* change a tree, and here a coarsening does, by discarding split resolution. None of them *helps*, exactly as expected, because the tree already finds BMI thresholds internally.

**Winner:** `raw21 + bmi_squared` is carried forward as the winning BMI encoding.

### 3.6 — Direct candidates & conditional contribution

Having screened individual blocks, we now (1) score the named candidate sets directly, (2) measure the conditional contribution of each retained block *given* the winning BMI encoding, and (3) use the conditional delta to finalize the headline pool.

In [12]:
# --- determine winning BMI encoding from bmi_results ---
bmi_ref_pr = bmi_results["raw21 (BMI raw, ref)"]["pr"]
bmi_pool = {
    "BMI (raw)":                       ([], []),
    "raw21 + bmi_squared":             (["BMI_squared"], []),
    "raw21 [BMI->capped]":             (["BMI_capped"], ["BMI"]),
    "raw21 [BMI->cat]":                (["BMI_cat"], ["BMI"]),
    "raw21 [BMI->who_onehot]+bmi_sq":  (WHO_OH + ["BMI_squared"], ["BMI"]),
}
bmi_winner_name = "BMI (raw)"; best_delta = 0.0
for s, r in bmi_results.items():
    if s == "raw21 (BMI raw, ref)": continue
    d = paired_delta(r["pr"], bmi_ref_pr)
    if d["retain"] and d["mean"] > best_delta:
        best_delta = d["mean"]; bmi_winner_name = s
add_cols_bmi, drop_cols_bmi = bmi_pool[bmi_winner_name]
raw21_plus_winning_bmi = [c for c in RAW21 if c not in drop_cols_bmi] + add_cols_bmi
print(f"Winning BMI encoding: {bmi_winner_name}")
print(f"  adds: {add_cols_bmi}, drops from raw21: {drop_cols_bmi}")
print()

# --- (1) Score named candidate sets on both instruments ---
winner_blocks_stage1 = [b for b, r in block_results["LogReg"].items() if r["retain"]]
union_pool_cols = list(dict.fromkeys(
    raw21_plus_winning_bmi + [c for b in winner_blocks_stage1 for c in BLOCKS[b]]
))
named_sets = {
    "raw21 + bmi_squared":  RAW21 + ["BMI_squared"],
    "union_pool":           union_pool_cols,
    "raw21 + all_additive": ALL_ADDITIVE,
}
direct_results = {}
print("=== Direct pool screening ===")
for sname, scols in named_sets.items():
    direct_results[sname] = {}
    for inst in INSTRUMENTS:
        direct_results[sname][inst] = screen(sname, scols, inst)
print()

# --- (2) Conditional contribution of each retained block given winning BMI ---
print("=== Conditional contribution (block | winning BMI encoding) ===")
conditional_deltas = {}
for bname in winner_blocks_stage1:
    base_cols = raw21_plus_winning_bmi
    full_cols = list(dict.fromkeys(raw21_plus_winning_bmi + BLOCKS[bname]))
    base_res = evaluate(f"cond_base", base_cols, "LogReg", log=False)
    full_res = evaluate(f"cond_{bname}", full_cols, "LogReg", log=False)
    d = paired_delta(full_res["pr"], base_res["pr"])
    conditional_deltas[bname] = d
    flag = "RETAIN" if d["retain"] else ("real, not worth" if d["real"] else "no robust gain")
    print(f"LogReg  {bname:20s} | winning BMI  d={d['mean']:+.4f}  sign={d['n_pos']}/5  "
          f"2SE={2*d['se']:.4f}  -> {flag}  [\u0394 per fold: {np.round(d['deltas'],4)}]")
print()

# --- (3) Build final headline_pool using conditional rule ---
final_winner_blocks = [b for b in winner_blocks_stage1
                       if conditional_deltas[b]["retain"] and conditional_deltas[b]["mean"] >= C_PRACTICAL]
final_engineered = [c for b in final_winner_blocks for c in BLOCKS[b]]
headline_pool = list(dict.fromkeys(raw21_plus_winning_bmi + final_engineered))
print(f"Final winner blocks (conditional rule): {final_winner_blocks or 'none'}")
print(f"Final headline_pool: {len(headline_pool)} columns")
print(f"  {headline_pool}")
print()

# --- (4) Side-by-side comparison table ---
print("=== Side-by-side comparison ===")
compare_sets = [
    ("raw21",            RAW21),
    ("raw21+bmi_sq",     RAW21 + ["BMI_squared"]),
    ("union_pool",       union_pool_cols),
    ("all_additive",     ALL_ADDITIVE),
]
print(f"{'Set':<22}  {'LR PR':>8}  {'LR \u0394':>7}  {'LGBM PR':>8}  {'LGBM \u0394':>7}")
for sname, scols in compare_sets:
    if sname == "raw21":
        lr_r = ref["LogReg"]; lgbm_r = ref["LightGBM"]
    else:
        lr_r = evaluate(sname, scols, "LogReg", log=False)
        lgbm_r = evaluate(sname, scols, "LightGBM", log=False)
    lr_d = paired_delta(lr_r["pr"], ref["LogReg"]["pr"])
    lgbm_d = paired_delta(lgbm_r["pr"], ref["LightGBM"]["pr"])
    print(f"{sname:<22}  {lr_r['pr_mean']:.4f}  {lr_d['mean']:+.4f}  {lgbm_r['pr_mean']:.4f}  {lgbm_d['mean']:+.4f}")

Winning BMI encoding: raw21 + bmi_squared
  adds: ['BMI_squared'], drops from raw21: []

=== Direct pool screening ===
LogReg    raw21 + bmi_squared                  PR 0.4189  d=+0.0148  sign 5/5  2SE=0.0018  -> RETAIN
LightGBM  raw21 + bmi_squared                  PR 0.4343  d=+0.0000  sign 0/5  2SE=0.0000  -> no robust gain
LogReg    union_pool                           PR 0.4189  d=+0.0148  sign 5/5  2SE=0.0018  -> RETAIN
LightGBM  union_pool                           PR 0.4339  d=-0.0004  sign 2/5  2SE=0.0008  -> no robust gain
LogReg    raw21 + all_additive                 PR 0.4203  d=+0.0161  sign 5/5  2SE=0.0016  -> RETAIN
LightGBM  raw21 + all_additive                 PR 0.4338  d=-0.0005  sign 3/5  2SE=0.0012  -> no robust gain

=== Conditional contribution (block | winning BMI encoding) ===
LogReg  composites           | winning BMI  d=-0.0000  sign=2/5  2SE=0.0001  -> no robust gain  [Δ per fold: [ 0.      0.     -0.0001 -0.     -0.0001]]

Final winner blocks (conditional 

#### Interpretation — Direct candidates & conditional contribution

**Key finding:** `BMI_squared` alone delivers essentially the *entire* LogReg feature-engineering gain. `raw21 + bmi_squared` reaches PR-AUC = 0.4189 (Δ = +0.0148 vs. raw21). Adding the `composites` block on top does not move it — `union_pool` (raw21 + bmi_squared + composites) also reaches 0.4189 (Δ = +0.0148, i.e. +0.0000 over bmi_squared alone). `all_additive` is marginally higher at 0.4203 (Δ = +0.0161), but its edge over the pool (+0.0014 vs. raw21 + bmi_squared) comes from `interactions`/`count_hurdle`, which were already sub-threshold in Stage 1.

**Conditional analysis:** The `composites` block contributes a *conditional* Δ = **−0.0000** (2/5 folds, per-fold deltas ≈ [0, 0, −0.0001, −0, −0.0001]) *given* the winning BMI encoding → **no robust gain**. Its Stage-1 gain of +0.0104 vs. raw21 was almost entirely the shared BMI non-linearity: `findrisc_lite` and `ada_risk_proxy` carry `BMI_cat`, which `BMI_squared` already captures more directly. Once `BMI_squared` is in the model the composites add nothing — the four linear-sum composites are collinear with raw columns, and the two `BMI_cat`-based ones are redundant with `BMI_squared`.

**Conclusion:** Headline pool = **raw21 + BMI_squared** (22 columns). No block clears the conditional bar, so nothing beyond `BMI_squared` enters. This is the parsimonious, rule-consistent result: the only engineered feature that survives for the linear model is the quadratic BMI term. LightGBM shows Δ ≈ 0 (or slightly negative) for every pool, consistent with the overall result.

| Set | LR PR-AUC | LR Δ | LGBM PR-AUC | LGBM Δ |
|---|---|---|---|---|
| raw21 | 0.4041 | +0.0000 | 0.4343 | +0.0000 |
| raw21 + bmi_sq | 0.4189 | +0.0148 | 0.4343 | +0.0000 |
| union_pool | 0.4189 | +0.0148 | 0.4339 | −0.0004 |
| all_additive | 0.4203 | +0.0161 | 0.4338 | −0.0005 |

### 3.7 Multiplicity surface (enumerated)

Naming the protections is not enough — the comparison count is put on the table so the reader can see
the multiplicity surface. NNo correction is applied as a gate: a formal pass (e.g. Benjamini-Hochberg) would rest on per-comparison p-values that n = 5 dependent folds cannot reliably supply; control is **structural**: domain-grouped blocks (not blind search),
**no free search over the 45 columns**, a factorised (not gridded) NB06 search, paired-direction
consistency rather than mean ± std overlap, only a few winners carried to NB07, and the single
unbiased test-set readout in NB08.

In [13]:
n_block   = len(BLOCKS) * len(INSTRUMENTS)
n_bracket = len(INSTRUMENTS)
n_bmi     = (len(bmi_sets) - 1) * 2          # 4 alternatives × both instruments
n_direct  = len(named_sets) * len(INSTRUMENTS)
n_cond    = len(winner_blocks_stage1)        # conditional re-test of Stage-1 winners (LogReg only)
n_total   = n_block + n_bracket + n_bmi + n_direct + n_cond
print(f"NB05 paired comparisons: blocks {n_block} + bracket {n_bracket} + BMI {n_bmi} "
      f"+ direct {n_direct} + conditional {n_cond} = {n_total}")
print("Gate = the paired CV-PR-AUC delta only; the descriptive metrics below never select.")

NB05 paired comparisons: blocks 8 + bracket 2 + BMI 8 + direct 6 + conditional 1 = 25
Gate = the paired CV-PR-AUC delta only; the descriptive metrics below never select.


### 3.8 Triangulation — descriptive, never gates

**Hard rule: retain/drop is *only* the paired CV delta above.** The three checks below *explain* a
delta or cross-check the linear side; none of them selects a feature. The danger they guard against is
back-door filtering.

1. **Single-feature PR-AUC, direction-aware** — in-sample sort. Protective features
   (`healthy_lifestyle`: high → lower risk) score *below* prevalence and look falsely signal-less, so
   we use `max(PR-AUC(f), PR-AUC(−f))`. Descriptive, **not** comparable to the grouped-CV deltas.
2. **L1 (Lasso) selection frequency over folds** — does the L1 path keep the same engineered columns
   the ablation retained? In-fold, stability reported, no full-data fit. Speaks only about LogReg.
3. **Out-of-fold permutation importance** on retained blocks — *which* column drove the delta.
   Engineered features are **exact functions of raw columns**, so importance is *redistributed*
   between a raw column and its engineered twin — a low score is not "useless".

In [14]:
# (1) direction-aware single-feature PR-AUC — DESCRIPTIVE in-sample sort, NOT a gate
prev = float(y_train.mean())
rows = []
for c in ADDITIVE_ENG + ["BMI_capped", "BMI_cat"] + WHO_OH:
    x = X_cand[c].astype(float).values
    rows.append((c, max(average_precision_score(y_train, x),
                         average_precision_score(y_train, -x))))
sf = pd.DataFrame(rows, columns=["feature", "sf_prauc"]).sort_values("sf_prauc", ascending=False)
print(f"prevalence floor = {prev:.4f}   (in-sample, direction-aware; NOT comparable to CV deltas)")
print(sf.to_string(index=False))

prevalence floor = 0.1393   (in-sample, direction-aware; NOT comparable to CV deltas)
                feature  sf_prauc
           BMI_x_HighBP  0.297808
              BMI_x_Age  0.290758
         ada_risk_proxy  0.275400
      comorbidity_count  0.271623
          findrisc_lite  0.261383
             BMI_capped  0.248763
            BMI_squared  0.247311
                BMI_cat  0.228441
           Age_x_HighBP  0.228219
      HighBP_x_HighChol  0.225349
            poor_health  0.217085
              ses_index  0.208519
           PhysHlth_log  0.205347
           PhysHlth_any  0.170235
      healthy_lifestyle  0.168016
            BMI_class_5  0.160626
            BMI_class_4  0.160479
           MentHlth_log  0.159276
            BMI_class_3  0.154809
            BMI_class_2  0.149892
            BMI_class_6  0.144255
           MentHlth_any  0.143919
healthcare_access_index  0.140723
            BMI_class_0  0.140390


#### Interpretation — Single-feature PR-AUC

The highest univariate scores go to `BMI_x_HighBP` (0.298) and `BMI_x_Age` (0.291) — interaction terms that concentrate a lot of raw signal by multiplying BMI with another strong predictor (HighBP, Age). That these ranked only "real, not worth" in the multivariate block ablation reveals the key distinction: **univariate in-sample PR-AUC ≠ multivariate conditional gain**. In the multivariate setting BMI and Age are already in the model, so the interaction term contributes only the deviation from their additive effect — far less than the naive in-sample score suggests.

Composite features (`ada_risk_proxy` 0.275, `comorbidity_count` 0.272, `findrisc_lite` 0.261) also score high because they additively bundle several prevalent risk factors — but in the multivariate context those components are already present individually. `healthcare_access_index` (0.141) and `healthy_lifestyle` (0.168) carry the least univariate signal: their constituent factors are less strongly associated with diabetes than BMI or Age, and the sign-correction (max over ±f) already absorbs their protective direction.

Tellingly, `BMI_squared` itself scores only 0.247 here — mid-pack, below every interaction and composite — yet it is the one engineered feature that survives the multivariate screen. The univariate ranking and the multivariate decision point in opposite directions, which is precisely why selection rests on the conditional CV delta and not on scores like these.

This analysis is only a plausibility check — it holds no veto over the ablation decision.

In [15]:
# Heuristic stability check at fixed C = 0.1, not tuned selection.
# FutureWarning suppressed: penalty API mid-transition (sklearn >= 1.8);
# TODO: migrate to l1_ratio=1 when on sklearn >= 1.10.
sel = pd.Series(0, index=ADDITIVE_ENG, dtype=int)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    warnings.simplefilter("ignore", UserWarning)
    for tr, _ in cv.split(X_cand, y_train, groups):
        Xtr = StandardScaler().fit_transform(X_cand.iloc[tr][ALL_ADDITIVE])
        l1 = LogisticRegression(penalty="l1", solver="saga", C=0.1, class_weight="balanced",
                                max_iter=2000, random_state=SEED).fit(Xtr, y_train.iloc[tr])
        coef = pd.Series(l1.coef_[0], index=ALL_ADDITIVE)
        sel[[f for f in ADDITIVE_ENG if abs(coef[f]) > 1e-8]] += 1
print("L1 selection frequency over 5 folds (heuristic stability check at C=0.1, not tuned selection):")
print((sel / 5).sort_values(ascending=False).to_string())

L1 selection frequency over 5 folds (heuristic stability check at C=0.1, not tuned selection):
MentHlth_any               1.0
PhysHlth_any               1.0
comorbidity_count          1.0
poor_health                1.0
Age_x_HighBP               1.0
BMI_x_Age                  1.0
BMI_x_HighBP               1.0
ses_index                  1.0
BMI_squared                1.0
HighBP_x_HighChol          1.0
PhysHlth_log               0.8
MentHlth_log               0.6
findrisc_lite              0.6
healthy_lifestyle          0.4
ada_risk_proxy             0.2
healthcare_access_index    0.2


#### Interpretation — L1 stability

Ten terms are selected in all five folds (frequency 1.0): `MentHlth_any`, `PhysHlth_any`, `comorbidity_count`, `HighBP_x_HighChol`, `poor_health`, `Age_x_HighBP`, `BMI_x_Age`, `BMI_x_HighBP`, `ses_index`, `BMI_squared`. `PhysHlth_log` follows at 0.8 (4/5). `BMI_squared` being held in every fold is consistent with its status as the one retained engineered term.

The rest of that list is the clearest demonstration in this notebook that **L1 stability ≠ usefulness**: most of these frequency-1.0 terms belong to blocks the ablation did *not* retain, for three different reasons. The **linear-sum composites** (`comorbidity_count` = HighBP + HighChol + Stroke + HeartDiseaseorAttack; `ses_index` = Education + Income) are exact linear combinations of raw columns — (near-)collinear with features already present, adding no new linear signal; L1 at C = 0.1 is not tight enough to drop them, it merely redistributes weight between a raw column and its composite twin. The **interactions** (`BMI_x_Age`, `BMI_x_HighBP`, `Age_x_HighBP`, `HighBP_x_HighChol`) are stable for a *different* reason — they carry genuine but small non-linear signal (the interactions block was "real, not worth", Δ = +0.0044); L1 keeps them, the ablation simply judged the gain below the practical floor. Sharpest of all, **`poor_health`** is selected in every fold yet its block (`flag`) produced *no* CV gain at all (Δ = −0.0004, 0/5): maximal L1 stability, zero multivariate value.

`findrisc_lite` (0.6) and `ada_risk_proxy` (0.2) add non-linearity only via `BMI_cat`, which `BMI_squared` already captures more directly, and `healthcare_access_index` is the weakest at 0.2 — its signal is largely subsumed by `CholCheck`, `AnyHealthcare` and `NoDocbcCost`.

This is a heuristic stability check at fixed C = 0.1, not a tuned selector: it explains which terms the L1 path holds stable, but gates no feature decision — only the paired CV delta does.

In [16]:
# (3) out-of-fold permutation importance on LogReg-retained blocks - DESCRIPTIVE.
from sklearn.inspection import permutation_importance

retained_lr = [b for b, r in block_results["LogReg"].items() if r["retain"]]
print("LogReg retained blocks (Stage 1):", retained_lr or "none")
if retained_lr:
    cols = RAW21 + [c for b in retained_lr for c in BLOCKS[b]]
    imp = pd.Series(0.0, index=cols)
    for tr, va in cv.split(X_cand, y_train, groups):
        est = make_logreg().fit(X_cand.iloc[tr][cols], y_train.iloc[tr])
        pi = permutation_importance(est, X_cand.iloc[va][cols], y_train.iloc[va],
                                    scoring="average_precision", n_repeats=5,
                                    random_state=SEED, n_jobs=-1)
        imp += pd.Series(pi.importances_mean, index=cols)
    print("\nOut-of-fold permutation importance (PR-AUC drop); engineered = exact functions of raw")
    print("columns, so importance redistributes between twins. Descriptive only.")
    print((imp / 5).sort_values(ascending=False).head(15).to_string())
else:
    print("Nothing retained for LogReg -> permutation importance skipped.")

LogReg retained blocks (Stage 1): ['composites']

Out-of-fold permutation importance (PR-AUC drop); engineered = exact functions of raw
columns, so importance redistributes between twins. Descriptive only.
GenHlth                    0.095937
ada_risk_proxy             0.064784
findrisc_lite              0.064090
Age                        0.058266
HighChol                   0.010198
comorbidity_count          0.009827
HighBP                     0.006343
CholCheck                  0.004037
HvyAlcoholConsump          0.003615
BMI                        0.002973
PhysActivity               0.002298
healthy_lifestyle          0.001603
healthcare_access_index    0.000804
PhysHlth                   0.000747
Income                     0.000709


#### Interpretation — Permutation importance

Permutation importance was computed on the Stage-1 retained block (`composites`) — i.e. on the feature set raw21 + composites, **without** `BMI_squared`. In this context `GenHlth` (0.096), `ada_risk_proxy` (0.065) and `findrisc_lite` (0.064) dominate: `ada_risk_proxy` and `findrisc_lite` appear important because they embed `BMI_cat`, the only BMI non-linearity present in this set — they carry the BMI curvature that `BMI_squared` would otherwise capture more directly.

This illustrates a key principle: **importance values are context-dependent and do not transfer across feature sets.** If `ada_risk_proxy` were added to the headline pool raw21 + bmi_squared, it would barely register, because `BMI_squared` already covers the BMI non-linearity directly and importance redistributes between correlated twins. The same mechanism reconciles an apparent contradiction with the L1 check: `ada_risk_proxy` ranks 2nd here (raw21 + composites, no BMI²) yet is selected in only 0.2 of folds by L1 (in all_additive, where BMI² *is* present). Same feature, different surrounding set — the interpretation changes accordingly.

`GenHlth` dominates as expected: it is the strongest single predictor here and loses weight to no engineered feature.

### 3.9 Selection-optimism spot check (nested vs. fixed, LogReg)

The `winners` set is selected on the train-CV and then reused as a *fixed* set in NB06, so NB06's CV
inherits selection optimism. Rather than only *naming* this, we **bound** it: the same L1 selector is
run two ways on the `all_additive` matrix — selected **once on the full training data** (leaks the
choice into every fold) vs. re-selected **inside each fold** (honest). The gap is the optimism the
fixed set carries. A small gap means the fixed `winners` set is safe to reuse for the seminar; the
frozen NB08 test set remains the only unbiased readout. (This is the pragmatic stand-in for full
nested selection.)

In [17]:
from sklearn.feature_selection import SelectFromModel

base_l1 = LogisticRegression(penalty="l1", solver="saga", C=0.1, class_weight="balanced",
                             max_iter=2000, random_state=SEED)

honest_pipe = ImbPipeline([("scaler", StandardScaler()),
                           ("select", SelectFromModel(base_l1)),
                           ("clf", LogisticRegression(class_weight="balanced",
                                                      max_iter=2000, random_state=SEED))])
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    warnings.simplefilter("ignore", UserWarning)
    honest = cross_validate(honest_pipe, X_cand[ALL_ADDITIVE], y_train, groups=groups,
                            cv=cv, scoring="average_precision", n_jobs=-1)["test_score"]

    Xs = StandardScaler().fit_transform(X_cand[ALL_ADDITIVE])
    support = SelectFromModel(base_l1).fit(Xs, y_train).get_support()
fixed_cols = list(np.array(ALL_ADDITIVE)[support])
optimistic = cross_validate(make_logreg(), X_cand[fixed_cols], y_train, groups=groups,
                            cv=cv, scoring="average_precision", n_jobs=-1)["test_score"]

print(f"fixed-selection (optimistic) PR-AUC : {optimistic.mean():.4f}")
print(f"nested-selection (honest)    PR-AUC : {honest.mean():.4f}")
print(f"selection optimism (fixed - nested) : {optimistic.mean() - honest.mean():+.4f}")
print("The small gap suggests limited selection optimism for this L1 check; NB08 remains the unbiased confirmation.")

fixed-selection (optimistic) PR-AUC : 0.4202
nested-selection (honest)    PR-AUC : 0.4202
selection optimism (fixed - nested) : -0.0000
The small gap suggests limited selection optimism for this L1 check; NB08 remains the unbiased confirmation.


#### Interpretation — Optimism spot check

Fixed-selection (optimistic): PR-AUC = 0.4202. Nested-selection (honest): PR-AUC = 0.4202. Selection optimism = −0.0000.

The measured optimism is practically zero. This does not mean selection optimism is absent — at n = 5 folds and the tight L1 selector at C = 0.1, the variance in column selection across folds is low, so fixed and nested selection are nearly identical. The gap **bounds** (does not eliminate) optimism for this L1 check; NB08 remains the only unbiased confirmation, since the test split was frozen before any project decision was made.

## 4. Assemble the NB05 output

Headline pool = raw-21 + winning BMI encoding + blocks that clear the conditional rule (conditional contribution given winning BMI ≥ C_PRACTICAL and ≥ 4/5 sign). The full 45-column matrix is
**saved but bound**: non-winners are not hard-deleted (the screen probes the linear/tree axis; a
non-winner might help a distance-based model in NB06), but NB06 may use the full set **only** as a
predefined named set or via an in-fold selector — never as an unbounded 45-column subset search.

In [18]:
X_cand.to_parquet(DATA_DIR / "X_train_candidates.parquet")

# build candidate_table for JSON (scores for the four compared sets)
candidate_table = {}
for sname, scols in compare_sets:
    if sname == "raw21":
        lr_r2 = ref["LogReg"]; lgbm_r2 = ref["LightGBM"]
    else:
        lr_r2 = direct_results.get(sname, {}).get("LogReg") or evaluate(sname, scols, "LogReg", log=False)
        lgbm_r2 = direct_results.get(sname, {}).get("LightGBM") or evaluate(sname, scols, "LightGBM", log=False)
    lr_d2 = paired_delta(lr_r2["pr"], ref["LogReg"]["pr"])
    lgbm_d2 = paired_delta(lgbm_r2["pr"], ref["LightGBM"]["pr"])
    candidate_table[sname] = {
        "LogReg_pr_mean": lr_r2["pr_mean"], "LogReg_delta": lr_d2["mean"],
        "LightGBM_pr_mean": lgbm_r2["pr_mean"], "LightGBM_delta": lgbm_d2["mean"],
    }

feature_sets = {
    "seed": SEED,
    "raw21": RAW21,
    "blocks": BLOCKS,
    "all_additive": ALL_ADDITIVE,
    "engineered_winners_stage1": winner_blocks_stage1,
    "engineered_winners_conditional": final_winner_blocks,
    "conditional_deltas": {b: {"mean": float(d["mean"]), "n_pos": d["n_pos"], "retain": d["retain"]}
                           for b, d in conditional_deltas.items()},
    "bmi_encoding_winner": bmi_winner_name,
    "headline_pool": headline_pool,
    "candidate_table": candidate_table,
    "full_candidate_columns": list(X_cand.columns),
    "parsimony_note": ("headline_pool uses the conditional rule: only blocks whose delta GIVEN the "
                       "winning BMI encoding clears both the 4/5 sign + 2SE test AND C_PRACTICAL=0.005. "
                       "If no block clears this conditional bar, pool = raw21 + winning_BMI_encoding only."),
    "decision_rule": {
        "sign_min_of_5": SIGN_MIN, "c_practical": C_PRACTICAL,
        "se_note": ("se=sd/sqrt(5) is optimistic under fold dependence (Bengio & Grandvalet 2004); "
                    "used as a lenient screening heuristic, backstopped by the >=4/5 sign rule and "
                    "the one-time NB08 test-set evaluation."),
    },
    "full_matrix_policy": ("Saved but BOUND: NB06 may use the full set only as a predefined named set "
                           "or via an in-fold selector, never as an unbounded subset search."),
}
with open(DATA_DIR / "feature_sets_nb05.json", "w") as f:
    json.dump(feature_sets, f, indent=2)
print("saved:", DATA_DIR / "X_train_candidates.parquet")
print("saved:", DATA_DIR / "feature_sets_nb05.json")
print(f"\nFinal headline_pool ({len(headline_pool)} cols):", headline_pool)

saved: c:\Users\hashe\Downloads\Diabetes-Prediction-Hasher\data\processed\X_train_candidates.parquet
saved: c:\Users\hashe\Downloads\Diabetes-Prediction-Hasher\data\processed\feature_sets_nb05.json

Final headline_pool (22 cols): ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex', 'GenHlth', 'Age', 'Education', 'Income', 'MentHlth', 'PhysHlth', 'BMI', 'BMI_squared']


## Results at a glance
| Feature set | LogReg PR-AUC | LogReg Δ | LightGBM PR-AUC | LightGBM Δ |
|---|---|---|---|---|
| raw21 (reference) | 0.4041 | — | 0.4343 | — |
| raw21 + bmi_squared | 0.4189 | +0.0148 | 0.4343 | +0.0000 |
| union_pool (raw21 + bmi_sq + composites) | 0.4189 | +0.0148 | 0.4339 | −0.0004 |
| raw21 + all_additive | 0.4203 | +0.0161 | 0.4338 | −0.0005 |

**Decision:** Headline pool = **raw21 + BMI_squared** (22 columns). `BMI_squared` is the only addition that survives the conditional rule (Δ ≥ C_PRACTICAL = 0.005 given the winning BMI encoding). The `composites` block passes Stage 1 (+0.0104 unconditional) but its conditional contribution given `BMI_squared` collapses to **−0.0000** (2/5 folds, no robust gain): its `BMI_cat` component duplicates the non-linearity already captured by `BMI_squared`, and the four linear-sum composites are collinear with raw columns. LightGBM gains nothing robustly from any engineered feature, consistent with the NB04-predicted 0.030 linear↔tree gap. The headline pool is persisted to `feature_sets_nb05.json`; the full 45-column candidate matrix is in `X_train_candidates.parquet`, accessible in NB06 only as a predefined named set or via an in-fold selector.

## Summary

**What was tested.** raw-21 (reference, per instrument) → `all_additive` bracket → four `+block` ablations on both instruments → BMI encoding swaps on both instruments → conditional contribution of retained blocks given the winning BMI encoding. Every set is logged as a named `feature_set` with its per-fold PR-AUC/ROC-AUC arrays in `outputs/results.csv`.

**How it was decided.** A single pre-specified rule (3.2): positive Δ in ≥ 4/5 folds *and* `mean(Δ) ≥ 2·SE(Δ)` (real), *and* `mean(Δ) ≥ 0.005` (worth it). The `2·SE` bar is an explicitly lenient heuristic — `SE=sd/√5` is optimistic because the folds are not independent (Bengio & Grandvalet 2004) — so the sign rule and NB08 are the real backstops. Filter/embedded diagnostics (single-feature PR-AUC, L1 stability, permutation importance) are **descriptive only and never gate**. The headline pool uses a **conditional rule**: only blocks whose delta *given the winning BMI encoding* clears the practical floor stay in.

**Rigor checks.** The multiplicity surface is enumerated (3.7), and selection optimism is *measured*, not just named, via a nested-vs-fixed L1 spot check (3.9).

**Output.** The winning BMI encoding was `BMI_squared`, and no block cleared the conditional bar, so the **headline pool = raw-21 + BMI_squared (22 columns)** — the quadratic BMI term is the only feature engineering that survives, and only for LogReg. The full 45-column matrix is also saved under a strict no-free-search policy; both are persisted to `feature_sets_nb05.json` for NB06. This matches the NB04-predicted 0.030 linear↔tree gap: LightGBM gains ≈ 0 from any engineered feature, while LogReg gains a single small but robust term.

## References

### Feature-engineering sources

- World Health Organization (2000). *Obesity: Preventing and Managing the Global Epidemic.* WHO Technical Report Series 894. — BMI cut-points.
- Tirosh, A., et al. (2011). Adolescent BMI trajectory and the risk of diabetes versus coronary disease. *New England Journal of Medicine, 364*(14), 1315–1325. — J-shaped BMI risk → `BMI_squared`.
- Mullahy, J. (1986). Specification and testing of some modified count data models. *Journal of Econometrics, 33*(3), 341–365. — hurdle models for zero-inflated counts.
- Lindström, J., & Tuomilehto, J. (2003). The diabetes risk score (FINDRISC). *Diabetes Care, 26*(3), 725–731. — `findrisc_lite`.
- Bang, H., et al. (2009). A patient self-assessment diabetes screening score. *Annals of Internal Medicine, 151*(11), 775–783. — `ada_risk_proxy`.
- Lloyd-Jones, D. M., et al. (2010). Defining and setting national goals for cardiovascular health promotion (Life's Simple 7). *Circulation, 121*(4), 586–613. — `healthy_lifestyle`.
- Andersen, R., & Newman, J. F. (1973). Societal and individual determinants of medical care utilization. *Milbank Memorial Fund Quarterly.* — `healthcare_access_index`.
- Idler, E. L., & Benyamini, Y. (1997). Self-rated health and mortality. *Journal of Health and Social Behavior, 38*(1), 21–37. — `poor_health`.
- Muhammad, M. A., Sani, J., & Ahmed, M. M. (2025). Exploring explainable machine learning for predicting and interpreting self-reported diabetes among Tennessee adults: insights from the 2023 BRFSS. *Journal of Primary Care & Community Health, 16.* doi:10.1177/21501319251400546 — top SHAP predictors (related work; BRFSS diabetes).
- Agardh, E., Allebeck, P., Hallqvist, J., Moradi, T., & Sidorchuk, A. (2011). Type 2 diabetes incidence and socio-economic position: a systematic review and meta-analysis. *International Journal of Epidemiology, 40*(3), 804–818. — `ses_index`.
- Landsberg, L., Aronne, L. J., Beilin, L. J., Burke, V., Igel, L. I., Lloyd-Jones, D., & Sowers, J. (2013). Obesity-related hypertension: pathogenesis, cardiovascular risk, and treatment. *Journal of Clinical Hypertension, 15*(1), 14–33. — `BMI_x_HighBP`.
- Janssen, I., Katzmarzyk, P. T., & Ross, R. (2005). Body mass index is inversely related to mortality in older people after adjustment for waist circumference. *Journal of the American Geriatrics Society, 53*(12), 2112–2118. — `BMI_x_Age` (age-modified adiposity).
- International Diabetes Federation (2006). *The IDF Consensus Worldwide Definition of the Metabolic Syndrome.* Brussels: IDF. — `HighBP_x_HighChol`.

### Methodology / statistics sources

- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer.
- Breiman, L., Friedman, J., Olshen, R., & Stone, C. (1984). *Classification and Regression Trees.* Wadsworth.
- Friedman, J. H. (2001). Greedy function approximation: a gradient boosting machine. *Annals of Statistics, 29*(5), 1189–1232.
- He, X., et al. (2014). Practical lessons from predicting clicks on ads at Facebook. *ADKDD '14.* (Cited for the idea that GBDT-derived feature transforms can help linear models; specific percentage claims not reproduced here.)
- Grinsztajn, L., Oyallon, E., & Varoquaux, G. (2022). Why do tree-based models still outperform deep learning on typical tabular data? *NeurIPS 2022, Datasets & Benchmarks.*
- Kuhn, M., & Johnson, K. (2019). *Feature Engineering and Selection.* Chapman & Hall/CRC.
- Bengio, Y., & Grandvalet, Y. (2004). No unbiased estimator of the variance of k-fold cross-validation. *JMLR, 5*, 1089–1105.
- Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432.